# NANO-GPT 50M — дообучение БЕЗ потери русского (тёплый старт)
Старая 50M уже говорит по-русски — мы НЕ учим её с нуля.
Переносим выученные эмбеддинги в новый словарь мультидиалогового корпуса
(154к диалогов, 95 млн символов) и дообучаем 2-3 часа.

Модель начинает с loss ~1.5 (не ~8 как с нуля), русский сохраняется,
+ связность реплик из мультидиалогов.

Твои веса 50M должны лежать на Драйве в MyDrive/nano-gpt (weights_big.npz + meta.json).

In [ ]:
# Клон и проверка GPU
!git clone https://github.com/lucifermornngstar52-cell/nano-gpt.git
%cd nano-gpt
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader || echo 'GPU НЕ ВКЛЮЧЕНА!'

In [ ]:
# Смонтировать Drive — оттуда возьмём старые веса 50M
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nano-gpt-50m-ft

In [ ]:
# Тёплый старт: переносим старые веса в новый словарь корпуса
%cd /content/nano-gpt
!cp /content/drive/MyDrive/nano-gpt/weights_big.npz weights_old.npz
!cp /content/drive/MyDrive/nano-gpt/meta.json meta_old.json
import os
os.environ.update(OLD_W='weights_old.npz', OLD_META='meta_old.json',
                  CORPUS='corpus_big.txt.gz', NEW_W='weights_big.npz')
!python finetune_warmstart.py

In [ ]:
# Дообучение (~2-3 ч на T4): START=1 — резюм со старыми знаниями, LR ниже
import os
os.environ.update(
    DATA='corpus_big.txt.gz',
    DM='640', NL='10', NH='10', CTX='256',
    STEPS='8000', BS='16', LR='3e-4',
    START='1',   # подхватит weights_big.npz из шага переноса
    CKPT='/content/drive/MyDrive/nano-gpt-50m-ft',
)
%cd /content/nano-gpt
!python train_torch.py

In [ ]:
# Чат-тест (на свежем чекпоинте с Драйва)
%cd /content/nano-gpt
import os, torch, numpy as np, json
os.environ.update(DM='640', NL='10', NH='10', CTX='256', DATA='corpus_big.txt.gz')
import train_torch as tt
tt.V = len(tt.vocab)
model = tt.NanoT().to(tt.dev)
z = np.load('/content/drive/MyDrive/nano-gpt-50m-ft/weights_big.npz')
with torch.no_grad():
    model.we.copy_(torch.tensor(z['we'])); model.wpe.copy_(torch.tensor(z['wpe']))
    for l in range(tt.NL):
        b = model.blocks[l]
        for nm in ('q','k','v','o','fc','fc2'): getattr(b, nm).copy_(torch.tensor(z[f'{l}{nm}']))
        b.ln1.weight.copy_(torch.tensor(z[f'{l}ln1g'])); b.ln1.bias.copy_(torch.tensor(z[f'{l}ln1b']))
        b.ln2.weight.copy_(torch.tensor(z[f'{l}ln2g'])); b.ln2.bias.copy_(torch.tensor(z[f'{l}ln2b']))
    model.lnf.weight.copy_(torch.tensor(z['lnfg'])); model.lnf.bias.copy_(torch.tensor(z['lnfb']))
model.eval()
@torch.no_grad()
def reply(q, temp=0.8):
    idx = torch.tensor([[tt.stoi.get(c,0) for c in ('В: '+q+'\nО:')[-256:]]], device=tt.dev)
    out = ''
    for _ in range(150):
        lg = model(idx[:, -256:])[0,-1]/temp
        k = torch.topk(lg, 8); ni = k.indices[torch.multinomial(torch.softmax(k.values,-1),1)]
        ch = tt.vocab[ni.item()]; out += ch
        if ch=='\n' and out.endswith('\n\n'): break
        idx = torch.cat([idx, ni.view(1,1)],1)
    return out.strip()
for q in ['как дела?','привет','расскажи о себе']:
    print('В:', q, '\nО:', reply(q), '\n')